# EEL6812 Project 1 — Binary Classification Network
**Cory Brynds**

In [ ]:
import pickle
import numpy as np
import os
import sys
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
    "figure.titlesize": 18,
})

plot_titles = True

## Plotting Utilities

In [ ]:
def save_fig_to_file(fig, save_path):
    os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)


def plot_losses(epochs, ce_losses, total_losses, title="Training Losses", save_path=None):
    fig, ax = plt.subplots()
    ax.plot(epochs, ce_losses, label="Cross-Entropy Loss", linewidth=1.5)
    ax.plot(epochs, total_losses, label="Total Loss (CE + L2)", linewidth=1.5)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title(title)
    ax.legend(loc="best")
    ax.grid(True, linestyle="--", alpha=0.6)
    save_fig_to_file(fig, save_path)


def plot_error_rates(epochs, train_errors, test_errors, title="Error Rates", save_path=None):
    fig, ax = plt.subplots()
    ax.plot(epochs, train_errors, label="Train Error", linewidth=1.5)
    ax.plot(epochs, test_errors, label="Test Error", linewidth=1.5)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Misclassification Rate")
    ax.set_title(title)
    ax.legend(loc="best")
    ax.grid(True, linestyle="--", alpha=0.6)
    save_fig_to_file(fig, save_path)


def plot_sweep_errors(runs, title="Error Rate Comparison", save_path=None):
    fig, (ax_train, ax_test) = plt.subplots(1, 2, sharey=True, figsize=(12, 5))

    for run in runs:
        ax_train.plot(run["epochs"], run["train_error_rates"], label=run["label"], linewidth=1.5)
        ax_test.plot(run["epochs"], run["error_rates"], label=run["label"], linewidth=1.5)

    ax_train.set_xlabel("Epoch")
    ax_train.set_ylabel("Misclassification Rate")
    ax_train.title("Training Error")
    ax_train.legend(loc="best")
    ax_train.grid(True, linestyle="--", alpha=0.6)

    ax_test.set_xlabel("Epoch")
    ax_test.title("Testing Error")
    ax_test.legend(loc="best")
    ax_test.grid(True, linestyle="--", alpha=0.6)

    fig.tight_layout()
    save_fig_to_file(fig, save_path)


def plot_frobenius_norms(runs, title="Frobenius Norm of Weights", save_path=None):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    for run in runs:
        ax1.plot(run["epochs"], run["w1_frobenius"], label=run["label"], linewidth=1.5)
        ax2.plot(run["epochs"], run["w2_frobenius"], label=run["label"], linewidth=1.5)

    ax1.set_xlabel("Epoch")
    ax1.set_ylabel(r"$\|W^{(1)}\|_F$")
    ax1.title(r"$W^{(1)}$ Frobenius Norm")
    ax1.legend(loc="best")
    ax1.grid(True, linestyle="--", alpha=0.6)

    ax2.set_xlabel("Epoch")
    ax2.set_ylabel(r"$\|w^{(2)}\|_2$")
    ax2.title(r"$w^{(2)}$ L2 Norm")
    ax2.legend(loc="best")
    ax2.grid(True, linestyle="--", alpha=0.6)

    fig.suptitle(title)
    fig.tight_layout()
    save_fig_to_file(fig, save_path)


def plot_all(history, title_prefix="", save_dir=None):
    epochs = list(range(1, len(history["ce_loss"]) + 1))

    loss_path = os.path.join(save_dir, "losses.png")
    plot_losses(epochs, history["ce_loss"], history["total_loss"],
                title=f"{title_prefix}Training Losses", save_path=loss_path)

    err_path = os.path.join(save_dir, "error_rates.png")
    plot_error_rates(epochs, history["train_error"], history["test_error"],
                     title=f"{title_prefix}Error Rates", save_path=err_path)

## Network Layers

In [ ]:
class LinearTransform(object):

    def __init__(self, W, b):
        self.W = W
        self.momentum_W = np.zeros_like(W)
        self.b = b
        self.momentum_b = np.zeros_like(b)

    def forward(self, x):
        self.x = x
        return x @ self.W.T + self.b

    def backward(
        self,
        grad_output,
        learning_rate=0.0,
        momentum=0.0,
        l2_penalty=0.0,
    ):
        batch_size = self.x.shape[0]
        grad_input = grad_output @ self.W
        self.grad_W = grad_output.T @ self.x / batch_size + l2_penalty * self.W
        self.grad_b = grad_output.mean(axis=0)
        self.momentum_W = momentum * self.momentum_W - learning_rate * self.grad_W
        self.momentum_b = momentum * self.momentum_b - learning_rate * self.grad_b
        self.W += self.momentum_W
        self.b += self.momentum_b
        return grad_input


class ReLU(object):

    def forward(self, x):
        self.x = x
        return np.maximum(0, x)

    def backward(self, grad_output):
        return grad_output * (self.x > 0)


class SigmoidCrossEntropy(object):

    def forward(self, x):
        self.output = 1 / (1 + np.exp(-x))
        return self.output

    def compute_ce_loss(self, y):
        clipped = np.clip(self.output, 1e-12, 1 - 1e-12)
        self.ce_loss = -y * np.log(clipped) - (1 - y) * np.log(1 - clipped)
        return np.mean(self.ce_loss)

    def backward(self, grad_output):
        self.ce_loss_grad = self.output - grad_output
        return self.ce_loss_grad

## Network Definition

In [ ]:
class Net(object):

    def __init__(self, input_dims, hidden_units):
        # I chose to use He initialization: scale by sqrt(2 / input dim), initialize biases to 0
        self.fc1 = LinearTransform(
            np.random.randn(hidden_units, input_dims) * np.sqrt(2.0 / input_dims),
            np.zeros(hidden_units))
        self.relu = ReLU()
        self.fc2 = LinearTransform(
            np.random.randn(1, hidden_units) * np.sqrt(2.0 / hidden_units),
            np.zeros(1))
        self.sigmoid_cross_entropy = SigmoidCrossEntropy()

    def forward(self, x):
        x = self.fc1.forward(x)
        x = self.relu.forward(x)
        x = self.fc2.forward(x)
        x = self.sigmoid_cross_entropy.forward(x)
        return x

    def backward(self, x, y, learning_rate, momentum, l2_penalty):
        grad = self.sigmoid_cross_entropy.backward(y)
        grad = self.fc2.backward(grad, learning_rate, momentum, l2_penalty)
        grad = self.relu.backward(grad)
        grad = self.fc1.backward(grad, learning_rate, momentum, l2_penalty)
        return grad

    def train(self, x_batch, y_batch, learning_rate, momentum, l2_penalty):
        prediction = self.forward(x_batch)

        ce_loss = self.sigmoid_cross_entropy.compute_ce_loss(y_batch)
        l2_loss = (l2_penalty / 2) * (np.sum(self.fc1.W**2) + np.sum(self.fc2.W**2))
        total_loss = ce_loss + l2_loss

        self.backward(x_batch, y_batch, learning_rate, momentum, l2_penalty)
        return ce_loss, total_loss

    def evaluate(self, x, y, l2_penalty=0.0):
        predictions = self.forward(x)

        ce_loss = self.sigmoid_cross_entropy.compute_ce_loss(y)
        l2_loss = (l2_penalty / 2) * (np.sum(self.fc1.W**2) + np.sum(self.fc2.W**2))
        total_loss = ce_loss + l2_loss

        predicted_labels = (predictions >= 0.5).astype(float)
        error_rate = np.mean(predicted_labels != y)
        return ce_loss, total_loss, error_rate

## Training Helpers for Parameter Sweeping

In [ ]:
def train_model(train_x, train_y, test_x, test_y, input_dims,
                hidden_units=15, learning_rate=0.01, momentum=0.8,
                l2_penalty=0.0001, batch_size=200, num_epochs=100, label=""):
    num_examples = train_x.shape[0]
    num_batches = num_examples // batch_size
    nnet = Net(input_dims, hidden_units)

    history = {
        "ce_loss": [],
        "total_loss": [],
        "train_error": [],
        "test_error": [],
        "w1_frobenius": [],
        "w2_frobenius": [],
    }

    for epoch in range(num_epochs):
        cumulative_ce_loss = 0.0
        cumulative_total_loss = 0.0

        for b in range(num_batches):
            start_index = b * batch_size
            end_index = start_index + batch_size
            x_batch = train_x[start_index:end_index]
            y_batch = train_y[start_index:end_index]

            ce_loss, total_loss = nnet.train(x_batch, y_batch, learning_rate, momentum, l2_penalty)
            cumulative_ce_loss += ce_loss
            cumulative_total_loss += total_loss
            print('\r  [{}] Epoch {:3d}, Avg.Loss = {:.4f}'.format(
                label, epoch + 1, cumulative_total_loss / (b + 1)), end='')

        train_ce_loss, train_total_loss, train_error = nnet.evaluate(train_x, train_y, l2_penalty)
        test_ce_loss, test_total_loss, test_error = nnet.evaluate(test_x, test_y, l2_penalty)

        print()
        print('Train Loss: {:.3f}  Train Error: {:.2f}%  '
              'Test Loss: {:.3f}  Test Error: {:.2f}%'.format(
                  train_total_loss, 100. * train_error,
                  test_total_loss, 100. * test_error))

        history["ce_loss"].append(train_ce_loss)
        history["total_loss"].append(train_total_loss)
        history["train_error"].append(train_error)
        history["test_error"].append(test_error)
        history["w1_frobenius"].append(np.linalg.norm(nnet.fc1.W, 'fro'))
        history["w2_frobenius"].append(np.linalg.norm(nnet.fc2.W))

    return history


def sweep_parameter(param_name, param_values, defaults, train_x, train_y,
                    test_x, test_y, input_dims, save_dir="results"):
    runs = []
    for val in param_values:
        config = defaults.copy()
        config[param_name] = val
        label = f"{param_name}={val}"
        print(f"\n{'='*60}")
        print(f"  Sweep: {label}")
        print(f"{'='*60}")

        history = train_model(
            train_x, train_y, test_x, test_y, input_dims,
            hidden_units=config["hidden_units"],
            learning_rate=config["learning_rate"],
            momentum=config["momentum"],
            l2_penalty=config["l2_penalty"],
            batch_size=config["batch_size"],
            num_epochs=config["num_epochs"],
            label=label,
        )

        epochs = list(range(1, config["num_epochs"] + 1))
        runs.append({
            "epochs": epochs,
            "error_rates": history["test_error"],
            "train_error_rates": history["train_error"],
            "w1_frobenius": history["w1_frobenius"],
            "w2_frobenius": history["w2_frobenius"],
            "label": label,
        })

    sweep_dir = os.path.join(save_dir, f"sweep_{param_name}")
    plot_sweep_errors(
        runs,
        title=f"Error Rates vs {param_name}",
        save_path=os.path.join(sweep_dir, "error_comparison.png"),
    )

    if param_name == "l2_penalty":
        plot_frobenius_norms(
            runs,
            title="Weight Norms vs L2 Penalty",
            save_path=os.path.join(sweep_dir, "frobenius_norms.png"),
        )

    return runs

## Data Loading & Preprocessing

In [ ]:
if sys.version_info[0] < 3:
    data = pickle.load(open('cifar_2class_py2.p', 'rb'))
    train_x = data['train_data']
    train_y = data['train_labels']
    test_x = data['test_data']
    test_y = data['test_labels']
else:
    data = pickle.load(open('cifar_2class_py2.p', 'rb'), encoding='bytes')
    train_x = data[b'train_data']
    train_y = data[b'train_labels']
    test_x = data[b'test_data']
    test_y = data[b'test_labels']

num_examples, input_dims = train_x.shape
num_examples_test, _ = test_x.shape
train_y = train_y.reshape(-1, 1).astype(np.float64)
test_y = test_y.reshape(-1, 1).astype(np.float64)

min_x = np.reshape(np.amin(train_x, axis=1), (num_examples, 1))
max_x = np.reshape(np.amax(train_x, axis=1), (num_examples, 1))
min_array = np.tile(min_x, input_dims)
max_array = np.tile(max_x, input_dims)
train_x = np.true_divide((train_x - min_array), (max_array - min_array))

min_x = np.reshape(np.amin(test_x, axis=1), (num_examples_test, 1))
max_x = np.reshape(np.amax(test_x, axis=1), (num_examples_test, 1))
min_array = np.tile(min_x, input_dims)
max_array = np.tile(max_x, input_dims)
test_x = np.true_divide((test_x - min_array), (max_array - min_array))

print(f"Training set: {train_x.shape}")
print(f"Test set:     {test_x.shape}")

## Default Training Run

In [ ]:
num_epochs = 100
batch_size = 200
num_batches = num_examples // batch_size
hidden_units = 15
learning_rate = 0.01
momentum = 0.8
l2_penalty = 0.0001

history = {
    "ce_loss": [],
    "total_loss": [],
    "train_error": [],
    "test_error": []
}

nnet = Net(input_dims, hidden_units)

for epoch in range(num_epochs):
    cumulative_ce_loss = 0.0
    cumulative_total_loss = 0.0

    for b in range(num_batches):
        start_index = b * batch_size
        end_index = start_index + batch_size
        x_batch = train_x[start_index:end_index]
        y_batch = train_y[start_index:end_index]

        ce_loss, total_loss = nnet.train(x_batch, y_batch, learning_rate, momentum, l2_penalty)
        cumulative_ce_loss += ce_loss
        cumulative_total_loss += total_loss
        print('\r[Epoch {}]    Avg.Loss = {:.3f}'.format(
            epoch + 1, cumulative_total_loss / (b + 1)), end='')

    train_ce_loss, train_total_loss, train_error = nnet.evaluate(train_x, train_y, l2_penalty)
    test_ce_loss, test_total_loss, test_error = nnet.evaluate(test_x, test_y, l2_penalty)

    print()
    print('    Train Loss: {:.3f}    Train Error: {:.2f}%'.format(train_total_loss, 100. * train_error))
    print('    Test Loss:  {:.3f}    Test Error:  {:.2f}%'.format(test_total_loss, 100. * test_error))

    history["ce_loss"].append(train_ce_loss)
    history["total_loss"].append(train_total_loss)
    history["train_error"].append(train_error)
    history["test_error"].append(test_error)

plot_all(history, "Default ", "results")

## Hyperparameter Sweeps

In [ ]:
defaults = {
    "hidden_units": 15,
    "learning_rate": 0.01,
    "batch_size": 200,
    "momentum": 0.8,
    "l2_penalty": 0.0001,
    "num_epochs": 100,
}

sweeps = {
    "hidden_units": [5, 10, 15, 20, 25, 30],
    "learning_rate": [0.001, 0.01, 0.03, 0.05, 0.1],
    "momentum": [0, 0.5, 0.8, 0.9],
    "batch_size": [10, 50, 100, 200, 500],
    "l2_penalty": [0, 0.01, 0.001, 0.0001]
}

for param_name, param_values in sweeps.items():
    sweep_parameter(param_name, param_values, defaults,
                    train_x, train_y, test_x, test_y, input_dims)